# train_and_save

Letakkan dataset di direktori kerja. Jalankan sel secara berurutan.

BASE_DIR menyesuaikan direktori kerja notebook. Notebook belum dieksekusi.

In [ ]:
import json, re, time, random, os, pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from transformers import BertTokenizerFast, BertForSequenceClassification, get_linear_schedule_with_warmup

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cpu"
BASE_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()



In [ ]:
def load_json(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)

def find_dataset(filename):
    candidates = [os.path.join(BASE_DIR, filename), os.path.join(os.getcwd(), filename), filename]
    for path in candidates:
        if os.path.isfile(path):
            return path
    raise FileNotFoundError(f"Tidak menemukan '{filename}' di {candidates}")

df1 = load_json(find_dataset("Sarcasm_Headlines_Dataset.json"))
df2 = load_json(find_dataset("Sarcasm_Headlines_Dataset_v2.json"))
df = pd.concat([df1, df2], ignore_index=True)[["headline", "is_sarcastic"]]
df.drop_duplicates(subset="headline", inplace=True)
df.dropna(inplace=True)

def clean(t):
    t = t.lower()
    t = re.sub(r"http\S+", " ", t)
    t = re.sub(r"[^a-z' ]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

df["headline"] = df["headline"].map(clean)
print(f"Total unique headlines: {len(df)} | sarcastic ratio: {df.is_sarcastic.mean():.3f}")

X_train, X_temp, y_train, y_temp = train_test_split(
    df.headline, df.is_sarcastic, test_size=0.2, random_state=SEED, stratify=df.is_sarcastic
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=SEED, stratify=y_temp
)
print(f"Train/Val/Test sizes: {len(X_train)}/{len(X_val)}/{len(X_test)}")



In [ ]:
VOCAB_SIZE = 15000
MAX_LEN = 30

from collections import Counter
counter = Counter()
for t in X_train:
    counter.update(t.split())
most_common = counter.most_common(VOCAB_SIZE - 2)
word2idx = {w: i + 2 for i, (w, _) in enumerate(most_common)}
word2idx["<PAD>"] = 0
word2idx["<UNK>"] = 1

def encode(text, max_len=MAX_LEN):
    ids = [word2idx.get(w, 1) for w in text.split()][:max_len]
    ids = ids + [0] * (max_len - len(ids))
    return ids

class SarcasmDS(Dataset):
    def __init__(self, texts, labels):
        self.X = torch.tensor([encode(t) for t in texts], dtype=torch.long)
        self.y = torch.tensor(list(labels), dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

train_ds, val_ds, test_ds = SarcasmDS(X_train, y_train), SarcasmDS(X_val, y_val), SarcasmDS(X_test, y_test)
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=128)
test_dl = DataLoader(test_ds, batch_size=128)
VOCAB = len(word2idx)



In [ ]:
class AdditiveAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.W = nn.Linear(hidden_dim, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)
    def forward(self, h):
        mu = torch.tanh(self.W(h))
        scores = self.v(mu).squeeze(-1)
        alpha = torch.softmax(scores, dim=-1)
        context = torch.sum(h * alpha.unsqueeze(-1), dim=1)
        return context, alpha

class HybridCNNBiLSTMAttention(nn.Module):
    def __init__(self, vocab_size, emb_dim=64, cnn_filters=64, lstm_hidden=32):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.conv = nn.Conv1d(emb_dim, cnn_filters, kernel_size=3, padding=1)
        self.lstm = nn.LSTM(emb_dim, lstm_hidden, bidirectional=True, batch_first=True)
        self.attn = AdditiveAttention(lstm_hidden * 2)
        self.fc1 = nn.Linear(cnn_filters + lstm_hidden * 2, 64)
        self.fc2 = nn.Linear(64, 1)
        self.dropout = nn.Dropout(0.3)
    def forward(self, x):
        e = self.emb(x)
        c = torch.relu(self.conv(e.transpose(1, 2)))
        c = torch.max(c, dim=2).values
        h, _ = self.lstm(e)
        ctx, _ = self.attn(h)
        combined = torch.cat([c, ctx], dim=1)
        z = self.dropout(torch.relu(self.fc1(combined)))
        return self.fc2(z).squeeze(-1)

def run_epoch(model, dl, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    loss_fn = nn.BCEWithLogitsLoss()
    with torch.set_grad_enabled(is_train):
        for X, y in dl:
            logits = model(X)
            loss = loss_fn(logits, y)
            if is_train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * len(y)
            all_preds.extend((torch.sigmoid(logits) > 0.5).long().tolist())
            all_labels.extend(y.long().tolist())
    return total_loss / len(dl.dataset), accuracy_score(all_labels, all_preds)

def train_model(model, epochs, lr):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for ep in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = run_epoch(model, train_dl, optimizer)
        val_loss, val_acc = run_epoch(model, val_dl)
        print(f"  epoch {ep}/{epochs} | train_loss {tr_loss:.4f} acc {tr_acc:.3f} "
              f"| val_loss {val_loss:.4f} acc {val_acc:.3f} | {time.time()-t0:.1f}s")

def evaluate(model, dl):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for X, y in dl:
            logits = model(X)
            preds.extend((torch.sigmoid(logits) > 0.5).long().tolist())
            labels.extend(y.long().tolist())
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds),
        "recall": recall_score(labels, preds),
        "f1": f1_score(labels, preds),
        "cm": confusion_matrix(labels, preds),
    }

print("\n=== Training Model A: CNN-BiLSTM-Attention ===")
model_a = HybridCNNBiLSTMAttention(VOCAB)
train_model(model_a, epochs=6, lr=1e-3)
res_a = evaluate(model_a, test_dl)

torch.save(model_a.state_dict(), os.path.join(BASE_DIR, "model_cnn_bilstm_attention.pt"))
with open(os.path.join(BASE_DIR, "vocab.pkl"), "wb") as f:
    pickle.dump({"word2idx": word2idx, "vocab_size": VOCAB, "max_len": MAX_LEN}, f)



In [ ]:
BERT_NAME = "bert-base-uncased"
BERT_MAX_LEN = 48
BERT_BATCH = 32
BERT_EPOCHS = 3
BERT_LR = 2e-5
BERT_DIR = os.path.join(BASE_DIR, "bert_sarcasm_model")
BERT_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n=== Fine-tuning {BERT_NAME} (device: {BERT_DEVICE}, {BERT_EPOCHS} epoch) ===")

bert_tokenizer = BertTokenizerFast.from_pretrained(BERT_NAME)
bert_model = BertForSequenceClassification.from_pretrained(BERT_NAME, num_labels=2).to(BERT_DEVICE)

class HeadlineDS(Dataset):
    def __init__(self, texts, labels):
        self.texts = list(texts)
        self.labels = [int(l) for l in labels]
    def __len__(self): return len(self.labels)
    def __getitem__(self, i): return self.texts[i], self.labels[i]

def bert_collate(batch):
    texts, labels = zip(*batch)
    enc = bert_tokenizer(list(texts), padding=True, truncation=True,
                         max_length=BERT_MAX_LEN, return_tensors="pt")
    enc["labels"] = torch.tensor(labels, dtype=torch.long)
    return enc

bert_train_dl = DataLoader(HeadlineDS(X_train, y_train), batch_size=BERT_BATCH, shuffle=True, collate_fn=bert_collate)
bert_val_dl = DataLoader(HeadlineDS(X_val, y_val), batch_size=64, collate_fn=bert_collate)
bert_test_dl = DataLoader(HeadlineDS(X_test, y_test), batch_size=64, collate_fn=bert_collate)

def run_epoch_bert(model, dl, optimizer=None, scheduler=None):
    train = optimizer is not None
    model.train() if train else model.eval()
    total_loss, preds, labels = 0.0, [], []
    with torch.set_grad_enabled(train):
        for step, batch in enumerate(dl, 1):
            batch = {k: v.to(BERT_DEVICE) for k, v in batch.items()}
            out = model(**batch)
            if train:
                optimizer.zero_grad()
                out.loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                if step % 100 == 0:
                    print(f"    batch {step}/{len(dl)} | loss {out.loss.item():.4f}")
            n = batch["labels"].size(0)
            total_loss += out.loss.item() * n
            preds.extend(out.logits.argmax(dim=-1).tolist())
            labels.extend(batch["labels"].tolist())
    return total_loss / len(labels), accuracy_score(labels, preds), preds, labels

bert_opt = torch.optim.AdamW(bert_model.parameters(), lr=BERT_LR, weight_decay=0.01)
bert_total_steps = len(bert_train_dl) * BERT_EPOCHS
bert_sched = get_linear_schedule_with_warmup(bert_opt, int(0.1 * bert_total_steps), bert_total_steps)

best_val_acc = 0.0
for ep in range(1, BERT_EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc, _, _ = run_epoch_bert(bert_model, bert_train_dl, bert_opt, bert_sched)
    val_loss, val_acc, _, _ = run_epoch_bert(bert_model, bert_val_dl)
    print(f"  epoch {ep}/{BERT_EPOCHS} | train_loss {tr_loss:.4f} acc {tr_acc:.3f} "
          f"| val_loss {val_loss:.4f} acc {val_acc:.3f} | {time.time()-t0:.0f}s")
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        bert_model.save_pretrained(BERT_DIR)
        bert_tokenizer.save_pretrained(BERT_DIR)
        print("    -> model terbaik (val acc) disimpan")

bert_model = BertForSequenceClassification.from_pretrained(BERT_DIR).to(BERT_DEVICE)
_, _, bert_preds, bert_labels = run_epoch_bert(bert_model, bert_test_dl)
res_bert = {
    "accuracy": accuracy_score(bert_labels, bert_preds),
    "precision": precision_score(bert_labels, bert_preds),
    "recall": recall_score(bert_labels, bert_preds),
    "f1": f1_score(bert_labels, bert_preds),
    "cm": confusion_matrix(bert_labels, bert_preds),
}



In [ ]:
with open(os.path.join(BASE_DIR, "metrics.pkl"), "wb") as f:
    pickle.dump({
        "cnn_bilstm_attention": {k: v for k, v in res_a.items() if k != "cm"},
        "bert": {k: v for k, v in res_bert.items() if k != "cm"},
    }, f)

print("\n=== Perbandingan (test set) ===")
print(f"{'Model':28s}{'Acc':>8s}{'Prec':>8s}{'Rec':>8s}{'F1':>8s}")
for name, m in [("CNN-BiLSTM-Attention", res_a), ("BERT (bert-base-uncased)", res_bert)]:
    print(f"{name:28s}{m['accuracy']*100:8.2f}{m['precision']*100:8.2f}{m['recall']*100:8.2f}{m['f1']*100:8.2f}")
print("\nConfusion matrix - HNN:\n", res_a["cm"])
print("\nConfusion matrix - BERT:\n", res_bert["cm"])

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import seaborn as sns
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    for ax, cm, title in zip(axes, [res_a["cm"], res_bert["cm"]], ["HNN (CNN-BiLSTM-Attention)", "BERT"]):
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                    xticklabels=["Non-Sarcastik", "Sarcastik"], yticklabels=["Non-Sarcastik", "Sarcastik"],
                    annot_kws={"size": 14})
        ax.set_xlabel("Prediksi"); ax.set_ylabel("Aktual"); ax.set_title(title)
    plt.tight_layout()
    plt.savefig(os.path.join(BASE_DIR, "confusion_matrix.png"), dpi=150)
    print("\nGambar disimpan: confusion_matrix.png")
except Exception as e:
    print("\n(Gambar confusion matrix dilewati:", e, ")")

print("\nSelesai. Sekarang jalankan: python -m streamlit run app.py")